In [2]:
import pandas as pd
import numpy as np
import joblib

train_df = joblib.load('../data/processed/train_processed.pkl')
val_df = joblib.load('../data/processed/val_processed.pkl')

print(train_df.shape, val_df.shape)

(472432, 434) (118108, 434)


In [3]:
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df['isFraud']

X_val = val_df[feature_cols]
y_val = val_df['isFraud']

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"\nFeature count: {len(feature_cols)}")

X_train: (472432, 431), y_train: (472432,)
X_val: (118108, 431), y_val: (118108,)

Feature count: 431


In [4]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, classification_report
import time

# scale_pos_weight accounts for class imbalance (roughly ratio of negative:positive)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

start = time.time()
xgb_model.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.1f}s")

scale_pos_weight: 27.46
Training time: 13.8s


In [5]:
y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
y_pred = xgb_model.predict(X_val)

auc = roc_auc_score(y_val, y_pred_proba)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"AUC-ROC: {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred))

AUC-ROC: 0.9026
Precision: 0.2316
Recall: 0.7168

Confusion Matrix:
[[104379   9665]
 [  1151   2913]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95    114044
           1       0.23      0.72      0.35      4064

    accuracy                           0.91    118108
   macro avg       0.61      0.82      0.65    118108
weighted avg       0.96      0.91      0.93    118108



In [6]:
xgb_model_v2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

xgb_model_v2.fit(X_train, y_train)

y_pred_proba_v2 = xgb_model_v2.predict_proba(X_val)[:, 1]
y_pred_v2 = xgb_model_v2.predict(X_val)

auc_v2 = roc_auc_score(y_val, y_pred_proba_v2)
precision_v2 = precision_score(y_val, y_pred_v2)
recall_v2 = recall_score(y_val, y_pred_v2)

print(f"AUC-ROC: {auc_v2:.4f}")
print(f"Precision: {precision_v2:.4f}")
print(f"Recall: {recall_v2:.4f}")

AUC-ROC: 0.9045
Precision: 0.8014
Recall: 0.3484


In [7]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba_v2)

# Find threshold where precision is closest to 0.90
target_precision = 0.90
idx = np.argmin(np.abs(precisions - target_precision))

print(f"At precision ≈ {precisions[idx]:.4f}, threshold = {thresholds[idx]:.4f}, recall = {recalls[idx]:.4f}")

# Also show a few other useful points
for p_target in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    idx = np.argmin(np.abs(precisions - p_target))
    if idx < len(thresholds):
        print(f"Precision {p_target}: threshold={thresholds[idx]:.4f}, actual precision={precisions[idx]:.4f}, recall={recalls[idx]:.4f}")

At precision ≈ 0.9000, threshold = 0.8507, recall = 0.2192
Precision 0.5: threshold=0.1690, actual precision=0.5000, recall=0.5094
Precision 0.6: threshold=0.2419, actual precision=0.5999, recall=0.4557
Precision 0.7: threshold=0.3316, actual precision=0.7000, recall=0.4134
Precision 0.8: threshold=0.4925, actual precision=0.7999, recall=0.3511
Precision 0.9: threshold=0.8507, actual precision=0.9000, recall=0.2192
Precision 0.95: threshold=0.9820, actual precision=0.9500, recall=0.0795


In [8]:
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)

print(f"Best F1 threshold: {thresholds[best_idx]:.4f}")
print(f"At this threshold -> Precision: {precisions[best_idx]:.4f}, Recall: {recalls[best_idx]:.4f}, F1: {f1_scores[best_idx]:.4f}")

# F2 score (weights recall 2x more than precision) - often more realistic for fraud
f2_scores = 5 * (precisions * recalls) / (4 * precisions + recalls + 1e-10)
best_f2_idx = np.argmax(f2_scores)

print(f"\nBest F2 threshold: {thresholds[best_f2_idx]:.4f}")
print(f"At this threshold -> Precision: {precisions[best_f2_idx]:.4f}, Recall: {recalls[best_f2_idx]:.4f}, F2: {f2_scores[best_f2_idx]:.4f}")

Best F1 threshold: 0.2748
At this threshold -> Precision: 0.6413, Recall: 0.4390, F1: 0.5212

Best F2 threshold: 0.0845
At this threshold -> Precision: 0.3394, Recall: 0.6361, F2: 0.5414


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Scaling done.")
print(X_train_scaled.shape, X_val_scaled.shape)

Scaling done.
(472432, 431) (118108, 431)


In [10]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

start = time.time()
lr_model.fit(X_train_scaled, y_train)
print(f"Training time: {time.time() - start:.1f}s")

/home/kislay01/projects/MLOps_FraudDetection/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training time: 115.7s


In [11]:
y_pred_proba_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
y_pred_lr = lr_model.predict(X_val_scaled)

auc_lr = roc_auc_score(y_val, y_pred_proba_lr)
precision_lr = precision_score(y_val, y_pred_lr)
recall_lr = recall_score(y_val, y_pred_lr)

print(f"AUC-ROC: {auc_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall: {recall_lr:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_lr))


AUC-ROC: 0.8322
Precision: 0.1285
Recall: 0.6946

Confusion Matrix:
[[94899 19145]
 [ 1241  2823]]
